In [ ]:
import os
import glob
import zipfile
import re
import numpy as np

# re = string and text manipulation

# glob = pattern matching tool, using like *txt

# Structure: UT/raw (for zips) and UT/ifgramStack (for extracted matrices)
PROJECT_NAME = "UT"

workspace_dir = os.path.abspath(os.path.join(os.getcwd(), ".."))
project_dir = os.path.join(workspace_dir, "data", PROJECT_NAME)

zip_dir = os.path.join(project_dir, "raw")
out_dir = os.path.join(project_dir, "inputs", "ifgramStack")

print(f"[*] Scanning {zip_dir} for HyP3 zip files...")
os.makedirs(out_dir, exist_ok=True)

zip_files = glob.glob(os.path.join(zip_dir, "*.zip"))
total_zips = len(zip_files)

if total_zips == 0:
    print(f"[!] FATAL: No .zip files found in {zip_dir}.")
    print(f"    Make sure your downloaded files are mapped to data/{PROJECT_NAME}/raw/")
else:
    print(f"[*] FOUND {total_zips} ZIP FILES.")
    print(f"[*] Extracting structurally to: {out_dir} \n")

    # Iterate through each zip
    for i, zf in enumerate(zip_files, 1):
        basename = os.path.basename(zf)

        # Parse Dates from Filename (HyP3 format: S1AA_20250805T123456_20250910T123456...)
        matches = re.findall(r'(\d{8})T\d{6}', basename)
        if len(matches) < 2:
            print(f"[{i}/{total_zips}] [!] Missing valid dates in filename: {basename}. Skipping.")
            continue

        ref_date, sec_date = matches[0], matches[1]
        pair_folder_name = f"{ref_date}_{sec_date}"

        # Create the specific pair directory MintPy needs
        pair_dir = os.path.join(out_dir, pair_folder_name)
        os.makedirs(pair_dir, exist_ok=True)

        print(f"[{i}/{total_zips}] Processing Pair: {pair_folder_name}")

        try:
            with zipfile.ZipFile(zf, 'r') as z:
                namelist = z.namelist()

                # Define exactly what 4 geometric/physics files we need + the config text file!
                required_targets = {
                    'Phase': ['unw_phase.tif'],
                    'Coherence': ['corr.tif', 'coh.tif'],
                    'DEM': ['dem.tif'],
                    'Incidence angle': ['lv_theta.tif', 'inc_map.tif'],
                    'Metadata': ['.txt']
                }

                # Extract only the targeted files into the pair directory
                for tag, suffixes in required_targets.items():
                    # For metadata, ensure we don't accidentally match another txt file by making sure it's the main file
                    if tag == 'Metadata':
                        matched_files = [f for f in namelist if f.endswith('.txt') and 'README' not in f and 'parameters' not in f]
                    else:
                        matched_files = [f for f in namelist if any(f.endswith(s) for s in suffixes)]

                    if matched_files:
                        target_file_in_zip = matched_files[0]
                        out_file_name = os.path.basename(target_file_in_zip)
                        out_file_path = os.path.join(pair_dir, out_file_name)

                        if not os.path.exists(out_file_path):
                            print(f"    -> Extracting {tag:15} | {out_file_name}")
                            with z.open(target_file_in_zip) as source, open(out_file_path, "wb") as target:
                                target.write(source.read())
                        else:
                            pass
                    else:
                        print(f"    [!] Missing {tag} file in {basename}")

        except zipfile.BadZipFile:
            print(f"    [!] FATAL: {basename} is corrupted. Please re-download.")

    print(f"\n[+] STACK ARCHITECTURE COMPLETE.")
    print(f"[*] Ready for MintPy config. View tree in: {out_dir}")

## Time-Series Setup (MintPy Template)
The extracted TIFFs are useless to MintPy until we write the **Configuration Template**. This file acts as the steering wheel for the MintPy engine, telling it exactly where the data is, what algorithms to use for atmospheric correction, and what reference point to anchor the entire time-series to.


cd F:\S1CL\M9_insar\data\UT; conda run --no-capture-output -n insar_env python -m mintpy.cli.smallbaselineApp mintpy_config.txt --start modify_network


In [ ]:
import os
import glob
from osgeo import gdal

# GDAL is the C-based workhorse for geospatial manipulation. We use it to align matrices.
gdal.UseExceptions()

PROJECT_NAME = "UT"
workspace_dir = os.path.abspath(os.path.join(os.getcwd(), ".."))
out_dir = os.path.join(workspace_dir, "data", PROJECT_NAME, "inputs", "ifgramStack")

print("[*] Calculating Absolute Common Intersection for all HyP3 matrices...")
unw_files = glob.glob(os.path.join(out_dir, "*", "*unw_phase.tif"))

min_x_list, max_y_list, max_x_list, min_y_list = [], [], [], []

for f in unw_files:
    ds = gdal.Open(f)
    gt = ds.GetGeoTransform()
    cols = ds.RasterXSize
    rows = ds.RasterYSize

    # Calculate Bounding Box coordinates
    ulx = gt[0]
    uly = gt[3]
    lrx = ulx + gt[1] * cols
    lry = uly + gt[5] * rows # gt[5] is negative pixel height

    min_x_list.append(ulx)
    max_y_list.append(uly)
    max_x_list.append(lrx)
    min_y_list.append(lry)
    ds = None # Release memory

# Mathematical intersection (Tightest possible bound that fits ALL images)
inter_ulx = max(min_x_list)
inter_uly = min(max_y_list) # ULY is highest Y. Minimum of maximums gives intersection.
inter_lrx = min(max_x_list)
inter_lry = max(min_y_list) # LRY is lowest Y. Maximum of minimums gives intersection.

print(f"[+] Common Bounds: UL=({inter_ulx:.4f}, {inter_uly:.4f}) | LR=({inter_lrx:.4f}, {inter_lry:.4f})")

# RUN THIS IN A NEW CELL TO DIAGNOSE
print(f"Calculated ULX: {inter_ulx}, LRX: {inter_lrx}")
print(f"Calculated ULY: {inter_uly}, LRY: {inter_lry}")

if inter_ulx >= inter_lrx or inter_lry >= inter_uly:
    print("[!] FATAL: No common intersection found! The stack contains files that do not overlap.")
    # Check if you have files from different regions or misplaced granules
else:
    print("[+] Intersection is valid. Checking for disk/path issues.")


In [ ]:
# ----------------------------------------------------------------------
# Align all unwrapped‑phase interferograms to the common intersection grid
# ----------------------------------------------------------------------
import numpy as np
from osgeo import gdal

# ----------------------------------------------------------------------
# 1️⃣  Derive target pixel size and projection from the first interferogram
# ----------------------------------------------------------------------
if not unw_files:
    raise RuntimeError("No interferogram files were found – check the 'out_dir' path.")
first_ds = gdal.Open(unw_files[0])
if first_ds is None:
    raise RuntimeError(f"Unable to open reference file: {unw_files[0]}")
gt = first_ds.GetGeoTransform()          # (originX, pixelW, rotX, originY, rotY, pixelH)
pixel_w, pixel_h = gt[1], gt[5]           # pixel_h is negative
proj_wkt = first_ds.GetProjection()
first_ds = None  # close

# ----------------------------------------------------------------------
# 2️⃣  Compute target raster dimensions that exactly cover the intersection
# ----------------------------------------------------------------------
cols = int((inter_lrx - inter_ulx) / abs(pixel_w))
rows = int((inter_uly - inter_lry) / abs(pixel_h))

if cols <= 0 or rows <= 0:
    raise RuntimeError(
        f"Computed raster size is invalid (cols={cols}, rows={rows}). "
        "Check intersection coordinates and pixel size."
    )

# ----------------------------------------------------------------------
# 3️⃣  Create a dummy reference grid (used only for its GeoTransform & CRS)
# ----------------------------------------------------------------------
ref_path = os.path.join(workspace_dir, "reference_grid.tif")
driver = gdal.GetDriverByName("GTiff")
ref_ds = driver.Create(ref_path, cols, rows, 1, gdal.GDT_Float32)
ref_ds.SetGeoTransform((inter_ulx, pixel_w, gt[2], inter_uly, gt[4], pixel_h))
ref_ds.SetProjection(proj_wkt)
ref_ds.GetRasterBand(1).WriteArray(np.zeros((rows, cols), dtype=np.float32))
ref_ds = None  # flush to disk

# ----------------------------------------------------------------------
# 4️⃣  Warp every interferogram onto the reference grid
# ----------------------------------------------------------------------
aligned_dir = os.path.join(
    workspace_dir, "data", PROJECT_NAME, "inputs", "aligned_ifgramStack"
)
os.makedirs(aligned_dir, exist_ok=True)

for fpath in unw_files:
    out_name = os.path.basename(fpath)
    out_path = os.path.join(aligned_dir, out_name)

    # gdal.Warp handles reprojection, resampling, and clipping to the target bounds
    gdal.Warp(
        out_path,
        fpath,
        format="GTiff",
        outputBounds=(inter_ulx, inter_lry, inter_lrx, inter_uly),  # (ULX, LRY, LRX, ULY)
        width=cols,
        height=rows,
        dstSRS=proj_wkt,
        resampleAlg=gdal.GRA_NearestNeighbour,   # safest for phase data
        srcNodata=None,
        dstNodata=0,
        multithread=True,
    )
    print(f"[+] Warped -> {out_path}")

# ----------------------------------------------------------------------
# 5️⃣  Verify that all warped files share identical size & GeoTransform
# ----------------------------------------------------------------------
print("\nVerification of aligned interferograms:")
for fpath in sorted(glob.glob(os.path.join(aligned_dir, "*unw_phase.tif"))):
    ds = gdal.Open(fpath)
    if ds is None:
        print(f"  !! Failed to open {fpath}")
        continue
    gt_aligned = ds.GetGeoTransform()
    print(
        f"  {os.path.basename(fpath)} : "
        f"{ds.RasterXSize}×{ds.RasterYSize}  GT={gt_aligned}"
    )
    ds = None

print("\n[+] All interferograms have been aligned to a common grid.")